# Week 1: Load and explore EEG

**Dataset:** PhysioNet EEG Motor Movement/Imagery (109 subjects, 64 channels, 160 Hz).
Runs 4, 8, 12 are *motor imagery*: the subject imagines opening/closing the left (T1) or right (T2) fist.

**Goal:** see the raw signal, its spectrum, and the classic motor-imagery effect: imagining a hand movement
*decreases* mu (8–12 Hz) and beta (13–30 Hz) power over the opposite side of the brain (ERD, event-related desynchronization).
Left hand → right motor cortex (C4). Right hand → left motor cortex (C3).

In [ ]:
import numpy as np
import mne
from mne.datasets import eegbci

mne.set_log_level("WARNING")
SUBJECT = 1
RUNS = [4, 8, 12]  # motor imagery: left vs right fist

## 1. Load raw data
Files download to `~/mne_data` on first run (a few MB per subject).

In [ ]:
paths = eegbci.load_data(SUBJECT, RUNS, update_path=True)
raw = mne.concatenate_raws([mne.io.read_raw_edf(p, preload=True) for p in paths])
eegbci.standardize(raw)  # 'C3..' -> 'C3'
raw.set_montage(mne.channels.make_standard_montage("standard_1005"))
raw

In [ ]:
raw.plot(duration=10, n_channels=16, scalings="auto");

## 2. Power spectrum
Look for the 1/f slope, an alpha/mu bump around 10 Hz, and line noise at 60 Hz (US recording).

In [ ]:
raw.compute_psd(fmax=80).plot(average=True);

## 3. Filter
Band-pass 1–40 Hz removes slow drift and line noise. Always filter the continuous signal, *before* epoching.

In [ ]:
raw.filter(1.0, 40.0)
raw.compute_psd(fmax=80).plot(average=True);

## 4. Epochs
Cut the signal into trials around each cue: 1 s before to 4 s after.

In [ ]:
events, _ = mne.events_from_annotations(raw, event_id=dict(T1=2, T2=3))
epochs = mne.Epochs(raw, events, dict(left=2, right=3), tmin=-1.0, tmax=4.0,
                    baseline=None, picks="eeg", preload=True)
epochs

## 5. ERD: time-frequency over C3 and C4
Percent power change vs. the 1 s before the cue. Blue = power drop (ERD).

In [ ]:
freqs = np.arange(6, 31)
for cls in ["left", "right"]:
    tfr = epochs[cls].compute_tfr("morlet", freqs=freqs, n_cycles=freqs / 2, picks=["C3", "C4"], average=True)
    for ch in ["C3", "C4"]:
        tfr.plot(picks=ch, baseline=(-1.0, 0.0), mode="percent", vlim=(-0.5, 0.5),
                 title=f"{cls} hand imagery — {ch}");

## Exercises

1. Does the ERD show up on the *opposite* side (left → C4, right → C3)? How clear is it for this subject?
2. Change `SUBJECT` to 2, 3, 4. How much does the effect vary between people? (This is why cross-subject BCI is hard.)
3. Find an eye blink in `raw.plot()`. Which channels show it most?
4. Write 3 sentences in the README progress log on what you learned.